In [4]:
import numpy as np
import os
import torch
import tempfile

from pathlib import Path
from datasets import load_dataset
from moviepy.video.io.VideoFileClip import VideoFileClip
from IPython.display import Video, display
from safetensors.torch import load_file

from src.config import Config
from src.model.virality_predictor import ViralityPredictor
from src.model.data_processor import DataProcessor

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using MPS.')
else:
    device = torch.device('cpu')
    print('Using CPU.')

config = Config()
checkpoint_dir = Path(config.checkpoint_path)
checkpoint_folders = sorted(
    [folder for folder in checkpoint_dir.iterdir() if folder.is_dir()])
latest_checkpoint = checkpoint_folders[-1] if checkpoint_folders else None
assert latest_checkpoint, f'No checkpoint found inside {checkpoint_dir}'

print(f'Loading model from: {latest_checkpoint}.')
model = ViralityPredictor(config)
state_dict = load_file(latest_checkpoint / 'model.safetensors')
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print('Loading dataset.')
dataset = load_dataset(config.dataset_id)['train']
print(f'Computing {config.p_virality_threshold} percentile thresholds.')
engagement_scores = np.array(dataset['engagement_score'])
velocity_scores = np.array(dataset['view_velocity_score'])
engagement_threshold = np.quantile(
    engagement_scores, config.p_virality_threshold)
velocity_threshold = np.quantile(velocity_scores, config.p_virality_threshold)

print(f'Engagement threshold: {engagement_threshold:.4f}')
print(f'Velocity threshold: {velocity_threshold:.4f}')

example_idx = np.random.randint(0, len(dataset))
example = dataset[example_idx]

print(f'\n=== Running inference on example {example_idx} ===')
print(f'Ground truth engagement score: {example["engagement_score"]:.4f}')
print(f'Ground truth velocity score: {example["view_velocity_score"]:.4f}')
ground_truth_viral = (
    example['engagement_score'] >= engagement_threshold and
    example['view_velocity_score'] >= velocity_threshold
)
print(f'Ground truth is_viral: {ground_truth_viral}')

processor = DataProcessor(config)

# Create a batch of size 1
batch = {
    key: [example[key]] if not isinstance(
        example[key], bytes) else [example[key]]
    for key in example.keys()
}
processed = processor.process_batch(batch)
processed_batch = {
    k: v.to(device) if isinstance(v, torch.Tensor) else v
    for k, v in processed.items()
}
inference_batch = {
    k: v for k, v in processed_batch.items() if k != 'labels'
}

with torch.no_grad():
    assert model, 'Model didn\'t load successfully.'
    output = model(**inference_batch, labels=None)
    logits = output['logits']
    print(logits)

pred_engagement, pred_velocity = logits[0].cpu().numpy()

print(f'\n=== Predictions ===')
print(f'Predicted engagement score: {pred_engagement:.4f}')
print(f'Predicted velocity score: {pred_velocity:.4f}')

pred_is_viral = (
    pred_engagement >= engagement_threshold and
    pred_velocity >= velocity_threshold
)
print(f'\nPredicted is_viral: {pred_is_viral}')

# Visualization
print(f'\n=== Summary ===')
print(f'Ground truth: {"VIRAL" if ground_truth_viral else "NOT VIRAL"}')
print(f'Prediction:  {"VIRAL" if pred_is_viral else "NOT VIRAL"}')
print(f'Match: {ground_truth_viral == pred_is_viral}')
print('\n=== Video ===')
video_bytes = example['video_bytes']
if video_bytes:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as temp_file:
        temp_file.write(video_bytes)
        temp_path = temp_file.name
    try:
        clip = VideoFileClip(temp_path)
        print(
            f'Video specs - Size: {clip.size}, Duration: {clip.duration:.2f}s, FPS: {clip.fps}')
        clip.close()
        display(Video(temp_path, embed=True))
    finally:
        os.unlink(temp_path)
else:
    print('No video bytes available')

Using MPS.
Loading model from: data/checkpoints/checkpoint-6.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.norm.bias                                                    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias    

Loading dataset.
Computing 0.95 percentile thresholds.
Engagement threshold: 0.3803
Velocity threshold: 24.7553

=== Running inference on example 1908 ===
Ground truth engagement score: 0.1411
Ground truth velocity score: 24.0583
Ground truth is_viral: False
tensor([[nan, nan]], device='mps:0')

=== Predictions ===
Predicted engagement score: nan
Predicted velocity score: nan

Predicted is_viral: False

=== Summary ===
Ground truth: NOT VIRAL
Prediction:  NOT VIRAL
Match: True

=== Video ===
Video specs - Size: [256, 256], Duration: 4.00s, FPS: 1.0


In [8]:
from datasets import load_dataset

from src.config import Config

config = Config()
expected_dims = [
    'author_follower_count',
    'author_following_count',
    'author_total_heart_count',
    'author_video_count',
    'author_friend_count',
    'duration',
    'width',
    'height',
    'aspect_ratio',
    'vq_score',
    'user_verified',
    'is_private',
    'is_ad',
    'share_enabled',
    'stitch_enabled',
    'day_of_week',
    'hour_of_day',
    'engagement_score',
    'view_velocity_score',
]
dataset = load_dataset('rodmosc/viral', split='train')
filtered_dataset = dataset.filter(
    lambda example: all(example.get(dim) is not None for dim in expected_dims)
)
print(f"original rows: {len(dataset)}")
print(f"filtered rows: {len(filtered_dataset)}")
filtered_dataset.push_to_hub(config.dataset_id)

original rows: 23309
filtered rows: 22899


Uploading the dataset shards:   0%|          | 0/6 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/rodmosc/viral/commit/f09ce5296cd2b48f0fbd2730606aae8e4fa0c945', commit_message='Upload dataset', commit_description='', oid='f09ce5296cd2b48f0fbd2730606aae8e4fa0c945', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/rodmosc/viral', endpoint='https://huggingface.co', repo_type='dataset', repo_id='rodmosc/viral'), pr_revision=None, pr_num=None)